<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day16_practice2_LSTM_%EA%B8%B0%EC%9A%B8%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# RNN의 약점: 문장이 길수록 '첫 단어'의 기울기 소실

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# 왜 RNN은 기울기 소실되나 - 단어가 1000개면 tanh를 1000번 계산. 역전파로 기울기를 1000번 곱하면 0에 수렴된다
def first_word_grad(model_cls, seq_len, hidden=32):
  torch.manual_seed(0)
  model = model_cls(16, hidden, batch_first=True) # 16입력차원, 은닉 32차원, 입력 텐서 형태(배치크기, 단어길이, 벡터차원)
  x = torch.randn(1, seq_len, 16, requires_grad=True) # 문장 1개, 단어 개수, 16차원 벡터, 기울기 계산
  out, _= model(x) # 매 걸음의 출력 h들
  out[0, -1].sum().backward() # 0번 문장의 마지막 h 벡터값을 다 더해서 스칼라 숫자 하나로 backward()에 넣어준다
  return x.grad[0,0].abs().mean().item() # 0번 문장의 0번째 단어의 기울기(배치크기, 단어길이, 벡터차원)를 16개 평균내서 하나의 대표값으로

print(f"{'길이':>6s} | {'RNN':>12s} | {'LSTM':>12s}")
for L in [5, 20, 50, 100]: # 단어 개수
  g_rnn = first_word_grad(nn.RNN, L)
  g_lstm = first_word_grad(nn.LSTM, L)
  print(f"{L:>6d} | {g_rnn:>12.2e} | {g_lstm:>12.2e}") # LSTM (옵션 미설정 결과)

    길이 |          RNN |         LSTM
     5 |     2.33e-02 |     7.72e-03
    20 |     1.37e-06 |     4.20e-06
    50 |     1.65e-14 |     1.60e-12
   100 |     6.98e-28 |     9.35e-23


In [9]:
# 셀 2. LSTM (Long Short-Term Memory) - 기억의 컨베이어 벨트
# c (셀 상태): 장기 기억 - 컨베이어 벨트, 곱하기 변환을 거의 안 거치고 직진한다 → 기울기가 안 죽는 고속도로
# h (은닉 상태): 단기 기억 - 지금 걸음의 작업 메모

# 장기 기억은 '게이트' 3개가 관리 - 게이트도 전부 '학습되는' 작은 신경망
# 잊을까? Forget Gate → 저장할까? Input Gate → 무엇을 저장할까? Candidate →       무엇을 출력할까? Output Gate : 4세트의 파라미터를 사용

lstm = nn.LSTM(16, 32, batch_first=True) # 입력16, 은닉32짜리 LSTM
x = torch.rand(1, 10, 16) # (문장1, 단어10, 각16차원벡터)
out, (h_n, c_n) = lstm(x)  # out:(1,10,32), h_n:(1,1,32), c_n:(1,1,32), # h (hidden state, 단기기억), c (cell state: 장기기억)

print("\nLSTM 반환 모양:")
print(f"out: {out.shape}")
print(f"h_n: {h_n.shape}")
print(f"c_n: {c_n.shape}")
n_rnn = sum(p.numel() for p in nn.RNN(16,32).parameters())
n_lstm = sum(p.numel() for p in lstm.parameters())
print(f"\n파라미터: RNN {n_rnn:,} vs LSTM {n_lstm:,}") # 정확히 4배: 게이트 3개 + 후보


LSTM 반환 모양:
out: torch.Size([1, 10, 32])
h_n: torch.Size([1, 1, 32])
c_n: torch.Size([1, 1, 32])

파라미터: RNN 1,600 vs LSTM 6,400


In [10]:
# 셀 3. LSTM에서 " 잊기 게이트를 활짝 열면 첫 단어를 더 오래 기억한다"
# 잊기 게이트가 1에 가까우면 벨트가 그대로 통과(보존) 0에 가까우면 치운다. 미학습 상태는 0.5
# bias 를 +2 해서 게이트를 열면 sigmoid(2)가 0.88이 돼서 1에 가까우니 그대로 통과, 장기기억 보존

def first_grad_with_open_gate(ssq_len, hidden=32, open_gate=False):
  torch.manual_seed(0)
  lstm2 = nn.LSTM(16, hidden, batch_first=True)

  if open_gate:  # 잊지 게이트를 열수록 = 벨트를 보존할수록 = 첫 단어를 더 오래 기억한다
    with torch.no_grad():
      # LSTM의 bias는 4개 게이트 몫이 '한 줄에 이어붙어' 저장된다. bias [입력게이트 | 잊기게이트 | 후보 | 출력게이트] 각 hidden(=32)칸씩, 총
      lstm.bias_ih_l0[hidden:2*hidden] += 2.0
      lstm.bias_hh_l0[hidden:2*hidden] += 2.0 # bias 를 +2 키우면 sigmoid(입력+2)가 커진다 → sigmoid(2)=0.88
      # Forget Gate의 영의 bias를 키우면 → sigmoid가 1에 가까워지고 → 과거 Cell State를 더 많이 유지한다. 장기기억 보존

  torch.manual_seed(1)
  x = torch.randn(1, ssq_len, 16, requires_grad=True) # (1, 100, 16) 문장1, 단어100, 각 16차원벡터
  out, _= lstm2(x) # (1, 100, 32)
  out[0, -1].sum().backward() # out(0, -1) 마지막 h
  return x.grad[0, 0].abs().mean().item() # x.grad:(1, 100, 16) → x.grad[0,0] 첫 단어 자리 기울기 크

print("\n[잊기 게이트를 열면 - 첫 단어의 기울기]")
for L in [50, 100]:
  closed = first_grad_with_open_gate(L, open_gate=False)
  opened = first_grad_with_open_gate(L, open_gate=True)
  print(f" 길이 {L:3d}: 게이트 반닫힘 {closed:.2e} → 게이트 열림 {opened:.2e}")



[잊기 게이트를 열면 - 첫 단어의 기울기]
 길이  50: 게이트 반닫힘 1.04e-12 → 게이트 열림 1.04e-12
 길이 100: 게이트 반닫힘 1.11e-23 → 게이트 열림 1.11e-23
